# Предсказание выживания на «Титанике»
MLP на PyTorch по курсовой работе. Выполняйте ячейки сверху вниз.
Файл `titanic.csv` положите рядом с ноутбуком.

### 1. Библиотеки и настройки
Фиксируем seed для повторяемости эксперимента.

In [1]:
from copy import deepcopy
import json
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display

SEED = 42
epochs = 200
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path.cwd() if (Path.cwd() / "titanic.csv").exists() else Path.cwd() / "titanic"
print("Устройство:", device)

Устройство: cpu


### 2. Исходные данные
Загружаем таблицу пассажиров.

In [2]:
frame = pd.read_csv(ROOT / "titanic.csv", sep=None, engine="python")
print("Число пассажиров:", len(frame))

Число пассажиров: 891


### 3. Предобработка
Добавляем размер семьи и признак одиночного путешествия. Из билета берём
буквенный префикс, из каюты — палубу. Возраст и стоимость стандартизируем;
пропуски заполняем средним по train, новые категории кодируем как −1.

In [3]:
class Preprocessor:
    FEATURES = ["Pclass", "Sex", "Age", "SibSp", "Parch", "FamilySize",
                "IsAlone", "Ticket", "Fare", "Cabin", "Embarked"]
    NUMERIC = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
    CATEGORICAL = ["Sex", "Ticket", "Cabin", "Embarked"]

    def _build_features(self, frame):
        data = frame[self.NUMERIC + self.CATEGORICAL].copy()
        for column in self.CATEGORICAL:
            data[column] = data[column].astype("string").str.strip().str.upper()
        data["Ticket"] = (data["Ticket"].str.replace(r"[^A-Z]", "", regex=True)
                          .replace("", "REG").fillna("REG"))
        data["Cabin"] = data["Cabin"].str.extract(r"([A-Z])", expand=False).fillna("UNK")
        for column in ["Sex", "Embarked"]:
            data[column] = data[column].replace("", "UNK").fillna("UNK")
        data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
        data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
        return data

    def fit_transform(self, frame):
        data = self._build_features(frame)
        self.means = data[["Age", "Fare"]].mean()
        filled = data[["Age", "Fare"]].fillna(self.means)
        self.scales = filled.std(ddof=0).replace(0, 1)
        self.categories = {
            column: {value: code for code, value in enumerate(data[column].unique())}
            for column in self.CATEGORICAL
        }
        return self.transform(frame)

    def transform(self, frame):
        data = self._build_features(frame)
        data[["Age", "Fare"]] = (
            data[["Age", "Fare"]].fillna(self.means) - self.means
        ) / self.scales
        for column in self.CATEGORICAL:
            data[column] = data[column].map(self.categories[column]).fillna(-1)
        return data[self.FEATURES].to_numpy(dtype=np.float32)

### 4. Разделение 70/20/10
Тест откладываем до конца. Статистики предобработки считаем только по train.

In [4]:
def split_data(frame):
    development, test = train_test_split(
        frame, test_size=0.1, stratify=frame["Survived"], random_state=SEED,
    )
    train, validation = train_test_split(
        development, test_size=2 / 9, stratify=development["Survived"], random_state=SEED,
    )
    return train, validation, test

train, validation, test = split_data(frame)
preprocessor = Preprocessor()
train_x = torch.from_numpy(preprocessor.fit_transform(train))
validation_x = torch.from_numpy(preprocessor.transform(validation)).to(device)
train_y = torch.tensor(train["Survived"].to_numpy(dtype=np.float32))
validation_y = torch.tensor(validation["Survived"].to_numpy(dtype=np.float32), device=device)
loader = DataLoader(TensorDataset(train_x, train_y), batch_size=32, shuffle=True)

print(f"Train: {len(train)}, validation: {len(validation)}, test: {len(test)}")
print("Признаки:", ", ".join(preprocessor.FEATURES))

Train: 623, validation: 178, test: 90
Признаки: Pclass, Sex, Age, SibSp, Parch, FamilySize, IsAlone, Ticket, Fare, Cabin, Embarked


### 5. Нейронная сеть
Два скрытых слоя: 16 и 8 нейронов. BCEWithLogitsLoss уже включает sigmoid.

In [5]:
class TitanicNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(11, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, features):
        return self.net(features).squeeze(-1)

model = TitanicNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(model)

TitanicNet(
  (net): Sequential(
    (0): Linear(in_features=11, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)


### 6. Обучение
Останавливаемся после 10 эпох без улучшения val loss минимум на 0.0001.

In [6]:
best_loss, stale_epochs, best_epoch = float("inf"), 0, 0
history = []
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(features), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    model.eval()
    with torch.no_grad():
        logits = model(validation_x)
        validation_loss = criterion(logits, validation_y).item()
        validation_accuracy = ((logits >= 0) == validation_y.bool()).float().mean().item()
    history.append({"epoch": epoch, "train_loss": total_loss / len(train),
                    "val_loss": validation_loss, "val_accuracy": validation_accuracy})
    if epoch == 1 or epoch % 10 == 0:
        print(f"Эпоха {epoch:3d}: loss={total_loss / len(train):.4f}, "
              f"val_loss={validation_loss:.4f}, val_accuracy={validation_accuracy:.4f}")
    if validation_loss < best_loss - 1e-4:
        best_loss, best_epoch, stale_epochs = validation_loss, epoch, 0
        best_state = deepcopy(model.state_dict())
    else:
        stale_epochs += 1
        if stale_epochs >= 10:
            print(f"Ранняя остановка на эпохе {epoch}")
            break


# Возвращаем веса лучшей эпохи.
model.load_state_dict(best_state)
print(f"Лучшая эпоха: {best_epoch}; val loss: {best_loss:.4f}")

Эпоха   1: loss=0.6644, val_loss=0.6584, val_accuracy=0.6180
Эпоха  10: loss=0.5029, val_loss=0.5368, val_accuracy=0.7360
Эпоха  20: loss=0.4343, val_loss=0.4707, val_accuracy=0.7809
Эпоха  30: loss=0.4208, val_loss=0.4529, val_accuracy=0.7921
Эпоха  40: loss=0.4130, val_loss=0.4400, val_accuracy=0.7978
Эпоха  50: loss=0.4067, val_loss=0.4453, val_accuracy=0.7978
Ранняя остановка на эпохе 50
Лучшая эпоха: 40; val loss: 0.4400


### 7. Результаты на тесте
Порог классификации — 0.5. Тест не участвовал в обучении и ранней остановке.

In [7]:
def predict(model, preprocessor, frame):
    device = next(model.parameters()).device
    features = torch.from_numpy(preprocessor.transform(frame)).to(device)
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(features)).cpu().numpy()

def prediction_table(frame, probabilities):
    result = pd.DataFrame(index=frame.index)
    if "PassengerId" in frame:
        result["PassengerId"] = frame["PassengerId"]
    if "Survived" in frame:
        result["ActualSurvived"] = frame["Survived"]
    result["SurvivalProbability"] = probabilities
    result["Survived"] = (probabilities >= 0.5).astype(int)
    return result

probabilities = predict(model, preprocessor, test)
labels = probabilities >= 0.5
metrics = {
    "seed": SEED, "device": str(device), "epochs_run": epoch,
    "best_epoch": best_epoch, "best_val_loss": best_loss,
    "split_sizes": {"train": len(train), "validation": len(validation), "test": len(test)},
    "test": {
        "accuracy": accuracy_score(test["Survived"], labels),
        "precision": precision_score(test["Survived"], labels, zero_division=0),
        "recall": recall_score(test["Survived"], labels, zero_division=0),
        "f1": f1_score(test["Survived"], labels, zero_division=0),
    },
}

display(pd.Series(metrics["test"], name="Тест").to_frame().round(4))

,Тест
accuracy,0.7667
precision,0.7059
recall,0.6857
f1,0.6957


### 8. Два пассажира из работы
Сравниваем мужчину из третьего класса и женщину из первого класса.

In [8]:
examples = pd.DataFrame([
    {"PassengerId": 1, "Pclass": 3, "Sex": "male", "Age": 25, "SibSp": 0,
     "Parch": 0, "Ticket": "123456", "Fare": 7.25, "Cabin": None, "Embarked": "S"},
    {"PassengerId": 2, "Pclass": 1, "Sex": "female", "Age": 30, "SibSp": 1,
     "Parch": 1, "Ticket": "PC 17599", "Fare": 100, "Cabin": "C85", "Embarked": "C"},
])
example_predictions = prediction_table(examples, predict(model, preprocessor, examples))
display(examples.join(example_predictions[["SurvivalProbability", "Survived"]]))

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,SurvivalProbability,Survived
0,1,3,male,25,0,0,123456,7.25,NaN,S,0.094223,0
1,2,1,female,30,1,1,PC 17599,100.00,C85,C,0.915543,1


### 9. Сохранение модели
Сохраняем лучшие веса, параметры предобработки, историю и метрики.

In [9]:
output_dir = ROOT / "artifacts"
output_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.cpu().state_dict(), output_dir / "model.pt")
# Сохраняем параметры, а не объект класса: файл работает и при импорте, и при CLI-запуске.
with (output_dir / "preprocess.pkl").open("wb") as file:
    pickle.dump(vars(preprocessor), file)
pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)
prediction_table(test, probabilities).to_csv(output_dir / "test_predictions.csv", index=False)
(output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

example_predictions.to_csv(output_dir / "examples_predictions.csv", index=False)
print("Результаты сохранены в папке artifacts.")

Результаты сохранены в папке artifacts.


Метрики могут отличаться от указанных в документе: исходное разбиение и веса
в нём не приведены. Здесь показан результат выполненного эксперимента.